In [5]:
from explore_import import  *
import ionbot_preprocess as io
import data_preprocess as dt
import hpp_checker as hpp
from Download_UnimodDB import *

from pyteomics import mass
import itertools 
from itertools import combinations
import time
import re
import ast
from scipy.spatial import distance
from gql import gql, Client
from gql.transport.aiohttp import AIOHTTPTransport
import asyncio
import plotly.graph_objects as go
warnings.simplefilter(action='ignore', category=FutureWarning)

In [6]:
#base directories

root="/project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery"
processed_dir=f"{root}/oui-discovery-vv-data/processed/20251015-oui-discovery-ionbot-results"
# get processed files
processed_paths=dt.list_files(processed_dir)
# leave only non-entrapment and open search, only search files
tmp = processed_paths.copy()
for parent, files in processed_paths.items():
    if '-closed' in parent or '-entrap' in parent:
        del tmp[parent]
        continue
    if not any( 'group-walk-output.csv' in f for f in files):
        del tmp[parent]
processed_paths = tmp

# where to save pickles
results_dir = f"{root}/oui-discovery-vv-data/pickles"

In [7]:
def Download_Unimod_Dict_names():
    unimod = Download_UnimodDB()
    condensed_unimod = {}
    for uni_id,df in unimod.groupby("unimod_id").__iter__():
        condensed_unimod[uni_id] = {}
        condensed_unimod[uni_id]['residues'] = "".join([_ for _ in df.residue if '-' not in _])
        condensed_unimod[uni_id]['mono_mass'] = [_ for _ in set(df.mono_mass)][0]
        condensed_unimod[uni_id]['code_name'] = [_ for _ in set(df.code_name)][0]
        condensed_unimod[uni_id]['full_name'] = [_ for _ in set(df.full_name)][0]
        
    return condensed_unimod

In [8]:
unimod = Download_Unimod_Dict_names()
unimod_df=pd.DataFrame.from_dict(unimod, orient='index')
#mark substitutions
unimod_df["isSubstitution"]=unimod_df.full_name.apply(lambda x: "substitution" in x)
unimod_df_subs=unimod_df[unimod_df["isSubstitution"]]

In [9]:
def get_peptide_position_io(s):
    match = re.search(r"\(?\(?(\d+)-(\d+)\)?\)?", s)    
    return (int(match.group(1)) , int(match.group(2))) if match else None

def filter_nested_intervals(intervals):
    if len(intervals)==1: return True
    # Sort by start ascending, and end descending to prioritize wider intervals
    #unpack the list
    intervals=[i[0] for i in intervals]
    sorted_intervals = sorted(intervals, key=lambda x: (x[0], -x[1]))
    result = []
    for i, (start_i, end_i) in enumerate(sorted_intervals):
        is_nested = False
        for j, (start_j, end_j) in enumerate(sorted_intervals):
            if i != j and start_j <= start_i and end_i <= end_j:
                is_nested = True
                break
        if not is_nested:
            result.append((start_i, end_i))
    result_bool=[i in result for i in intervals]
    return result_bool
def filter_short_overlaps(i,overlapping_intervals,minimal_length):
    indexes,intervals = overlapping_intervals
    for indx,interval in zip(indexes,intervals):
        if i in indx and len(indx)==1:
            return True
        elif i in indx and len(indx)>1:
            return abs(interval[0]-interval[1])>=minimal_length
        else:
            continue

In [10]:
#correct manualy where best PSM and nan mass

unimod_fix={"9999989]":"1680]", #	1307.512681
           "999928]":"536]",
           "9999111]":"1427]", # 338.084912
           "9999248]":"1450]", # 1021.359809
           "9999986]":"1614]", #	948.343430
           "999990]":"60]",
           "9999936]":"1765]",
           "9999317]":"1763]", #1095.396588
           "9999575]":"285]", # 155.004099
           "9999833]":"1378]", # 956.322026
           "9999365]":"1599]",
           "9999805]":"1691]",
           "9999571]":"65]",
           "9999471]":"41]", #	162.052824
           "9999800]":"1377]", #	794.269203
           "9999558]":"1413]",
           "9999481]":"1434]",
           "9999212]":"1935]",
           "9999211]":"1441]",
           "99991]":"365]",
           "9999529]":"1368]", #	62.063875
           "9999122]":"149]", # 656.227613
           "9999999]":"1654]",
           "9999761]":"1596]",
           "9999363]":"1599]",
           "9999297]":"1932]",
           "9999292]":"1936]",
           "9999210]":"1441]",
           "9999126]":"1786]",
           "9999124]":"149]",
           "9999532]":"298]", # 	17.034480
           "999924]":"136]", # 104.026215
           "9999747]":"1376]", # 632.216379
           "9999869]":"1652]", #1161.454772
            "9999104]":"408]", #148.037173
            "9999117]":"793]", #365.132196
            "9999894]":"1760]", #	1022.380210
            "99991]":"365]", #105.021464
            "9999182]":"156]", #	860.327386
            "9999667]": "1641]", #	1093.380938,
            "9999675]": "1618]", # 	964.338345,
            "99995]": "866]", #	115.066700
            "999935]" : "366]", # 	2.988261
            "999972]":"510]",
            "999920]":"763]",
            "9999564]":"59]",
            "999977]":"1291",
            "99993]":"364]",
            "9999219]":"512]",
            "99991044]":"295]",
            "9999528]":"284]",
            "999911]":"1372]",
            "999915]":"56]",
            "9999581]":"1414]",
            "9999535]":"329]",
            "999971]":"199]",
            "9999178]":"152]",
            "9999493]":"143]",
            "9999315]":"162]",
            "9999542]":"212]",
            "9999561]":"428]",
            "9999102]":"54]",
            "999923]":"136]",
           } 
unimod_fix_mass={"9999989]":1307.512681,
                "9999111]":338.084912,
           "9999248]":1021.359809,
           "9999986]":948.343430,
                "9999317]":1095.396588,
           "9999575]":155.004099,
           "9999833]":956.322026,
                "9999471]":162.052824,
           "9999800]":794.269203,
                "9999529]":62.063875,
           "9999122]":656.227613,
                "9999532]":17.034480,
           "999924]":104.026215,
           "9999747]":632.216379,
           "9999869]":1161.454772,
            "9999104]":148.037173,
            "9999117]":365.132196,
            "9999894]":1022.380210,
            "99991]":105.021464,
            "9999182]":860.327386,
            "9999667]": 1093.380938,
            "9999675]": 964.338345,
               "99995]"  :	115.066700,
                 "999935]" : 2.988261,
                 "999972]":34.063117,
                 "9999571":104.041151,
                 "999920]":	142.039317,
                 "9999564]":59.036279,
                 "999977]":	34.068961,
                 "999990]":	127.099714,
                 "99993]":	111.041593,
                 "9999219]":324.105647,
                 "99991044]":146.057909,
                 "9999528]":16.028204,
                 "999911]":	44.017274,
                 "999915]":	45.029395,
                 "9999581]":54.113505,
                 "9999535]":18.037835,
                 "999971]":	32.056407,
                 "9999178]":714.269478,
                 "9999493]":	406.158745,
                 "9999315]":	972.283547,
                 "9999542]":90.084148,
                 "9999561]":	283.045704,
                 "9999102]":	176.032088,
                 "999923]":	104.026215,
                }

In [11]:
def get_modpep(row,pep_col="database_peptide",mod_col="modifications",mod_mas_col="modifications_masses",mode="spectrum_utils"):
    """
    modes: usi 
            '[UNIMOD:1]-AAK[UNIMOD:381]ALDRHQAHLC[UNIMOD:4]VLASNC[UNIMOD:4]DEPMYVK'
            spectrum_utils
            '[+42.010565]-AAK[+14.96328]ALDRHQAHLC[+57.021464]VLASNC[+57.021464]DEPMYVK'
            custom
            
    """
    #print(row)
    peptide=list(row[pep_col])
    modifications=row[mod_col]
    modifications_masses=row[mod_mas_col]
    #print(row.modifications)
    if modifications=="Unmodified": return row["database_peptide"]
    if mode=="spectrum_utils" or mode=="custom":
        #unimod_id=[mods.split("_or_")[0].split(']')[1].split('[')[0] if "_or_" in mods 
        #           else mods.split(']')[1].split('[')[0]  
        #           for mods in modifications.split("||")]
        mod_id=[mods[2] for mods in modifications_masses]
    elif mode=="usi":
        mod_id=[mods.split("_or_")[0].split("|")[1].split("]")[0][1:] if "_or_" in mods 
               else mods.split("|")[1].split("]")[0][1:]  
               for mods in modifications.split("||") ]
    #positions=[int(mods.split("_or_")[0].split("|")[0]) if "_or_" in mods 
    #           else int(mods.split("|")[0])  
    #           for mods in modifications.split("||")]
    positions=[mods[0] for mods in modifications_masses]
    
    i=0
    for pos,modid in sorted(zip(positions, mod_id), key=lambda pair: pair[0]): #sort by position
        if mode=="spectrum_utils":
            sign="+" if float(modid)>=0 else "-"
            t=f"[{sign}{modid}]"
        elif mode=="usi":
            t=f"[UNIMOD:{modid}]"
        elif mode=="custom":
            t=f"({modid})"
        if pos==0: t=t+"-"
        peptide.insert(pos+i,t)
        i+=1 #to correct the position
        #print(t)
    pep="".join(peptide)
    pep=pep.replace("--","-")
    return pep

In [12]:
def sankey_stepwise(df, column_name,figsave_folder,wildcard):
    steps = df.index.tolist()
    source_steps = steps[:-1]
    target_steps = steps[1:]
    
    source_labels = [f'{column_name.upper()}:{s}' for s in source_steps]
    target_labels = [f'{column_name.upper()}:{s}' for s in target_steps]
    
    all_labels = list(pd.unique(source_labels + target_labels))
    label_map = {l: i for i, l in enumerate(all_labels)}
    
    values = [df.loc[target, column_name] for target in target_steps]

    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=all_labels,
        ),
        link=dict(
            source=[label_map[s] for s in source_labels],
            target=[label_map[t] for t in target_labels],
            value=values
        )
    )])

    fig.update_layout(title_text=f"Filtering in {column_name.upper()} steps", font_size=10)
    fig.write_image(f"{figsave_folder}S_{wildcard}_hpp_countdown_{column_name}.svg")
    fig.show()

## Check non-canonical proteins identifications against HPP guidlines

In [41]:
def track_hpp_countdown(countdown_dict, newkey, pepdf, psmdf):
    countdown_dict["psm"][newkey]=len(psmdf[psmdf.database_peptide.isin(pepdf.database_peptide.tolist())])
    countdown_dict["pep"][newkey]=len(pepdf)
    countdown_dict["prot"][newkey]=len(pepdf.leadprot.unique())
    return countdown_dict
    
def is_goodlength(df,min_length=9):
    '''Filteres out peptides with length shorter than minimal.'''
    return df[df.peptide_length>=min_length]

def is_nonnested(df):
    '''Filteres out sub-peptides. Apply only on proteotypic peptides.'''
    df["NonNested"] = df.groupby("leadprot")["peptide_position"].transform(
        lambda intervals: filter_nested_intervals(intervals.values)
    )
    return df[df["NonNested"]]

def mark_overlaps(group,minimal_length=18):
    intervals = group.peptide_position.values
    if len(intervals) == 1:
        group["overlap_intervals"] = [True] * len(group)
    else:
        #unpack the list
        intervals=[i[0] for i in intervals]
        overlapping_intervals = hpp.find_max_length_in_overlap_groups(intervals)
        good_overlap = [
            filter_short_overlaps(i, overlapping_intervals, minimal_length) 
            for i in range(len(group))
        ]
        group["overlap_intervals"] = good_overlap
    return group

def is_nonoverlap(df):
    df = df.groupby("leadprot", group_keys=False).apply(mark_overlaps)
    return df[df.overlap_intervals]
    
def check_2_pep(group):
    return len(group)>=2

def is_2pep(df):
    df["isProt2Pep"]=df.leadprot.apply(lambda x: check_2_pep(df.groupby("leadprot").get_group(x)))
    return df[df.isProt2Pep]

In [18]:
#track how many we loose with each HPP criteria
hpp_countdown={"psm":{},"pep":{},"prot":{}}

In [16]:
#load the data
comb_datasets=pd.DataFrame()
for dataset_name,subdict in combined_first_datasets.items():
    df=subdict["openprot"]
    df["dataset"]=dataset_name
    comb_datasets=pd.concat([comb_datasets,df])
comb_datasets.reset_index(drop=True,inplace=True)

#add peptide length and parse peptide position
comb_datasets["peptide_length"]=comb_datasets.database_peptide.apply(lambda x: len(x))
comb_datasets["peptide_position"]=comb_datasets.proteins.apply(lambda x: [get_peptide_position_io(s) for s in x.split("||")])

#select only proteotypic non-canonical peptides
comb_datasets_ncun=comb_datasets[comb_datasets.peptide_class=="unique_to_Noncanon"]

In [19]:
#track
hpp_countdown["psm"]["original"]=len(comb_datasets_ncun)
hpp_countdown["pep"]["original"]=len(comb_datasets_ncun.drop_duplicates("database_peptide"))
hpp_countdown["prot"]["original"]=len(comb_datasets_ncun.leadprot.unique())

### Peptide level

In [42]:
comb_datasets_ncun_pepfilt=comb_datasets_ncun.copy(deep=True)

#Go to peptide level - leave 1 PSM per peptide
comb_datasets_ncun_pepfilt.drop_duplicates("database_peptide",inplace=True)

#Filter out peptides <9 aa long
comb_datasets_ncun_pepfilt=is_goodlength(comb_datasets_ncun_pepfilt)

#track <9aa
hpp_countdown=track_hpp_countdown(hpp_countdown, "<9aa", comb_datasets_ncun_pepfilt, comb_datasets_ncun)

#Filter out nested peptides
comb_datasets_ncun_pepfilt=is_nonnested(comb_datasets_ncun_pepfilt)

#track nested
hpp_countdown=track_hpp_countdown(hpp_countdown, "nested", comb_datasets_ncun_pepfilt, comb_datasets_ncun)

#For overlapping peptides, check the total extent length is >=18
comb_datasets_ncun_pepfilt=is_nonoverlap(comb_datasets_ncun_pepfilt)

#track <18aa
hpp_countdown=track_hpp_countdown(hpp_countdown, "<18aa", comb_datasets_ncun_pepfilt, comb_datasets_ncun)

#Filter out peptides, that are non-unqiely maped to peptide variants

#Input to the list into https://hppportal.net/toolchecker.html
#for pep in comb_datasets_ncun.database_peptide:
#    print(pep)
#comb_datasets_ncun_pepfilt["database_peptide"].to_csv("nc_uniqness_check.csv", index=False, header=False)
#read the output
uniqness_checker=pd.read_csv("nc_uniqness_checker_res.csv")
uniqness_checker["NoMatch"]=uniqness_checker['Subject Accession']=='NO MATCH'
#mark uniqness
comb_datasets_ncun_pepfilt=comb_datasets_ncun_pepfilt.merge(uniqness_checker[["Peptide","NoMatch"]],left_on="database_peptide",right_on="Peptide",how="left")
comb_datasets_ncun_pepfilt=comb_datasets_ncun_pepfilt[comb_datasets_ncun_pepfilt.NoMatch!=False]

#track uniqness
hpp_countdown=track_hpp_countdown(hpp_countdown, "uniqness", comb_datasets_ncun_pepfilt, comb_datasets_ncun)

#check for unique 2 peptides, filterout if not
comb_datasets_ncun_pepfilt=is_2pep(comb_datasets_ncun_pepfilt)

#track min 2 proteotypic peptides per protein
hpp_countdown=track_hpp_countdown(hpp_countdown, "2pep1", comb_datasets_ncun_pepfilt, comb_datasets_ncun)

/tmp/ipykernel_3067569/913464691.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("leadprot", group_keys=False).apply(mark_overlaps)


### PSM level

#### Proteotypic peptides with isoSAAV explained by gnomAD-SAAV

In [45]:
#load data from gnomAD analysis
gnomAD_qgr_res_uniq = pickle.load(open("./../gnomAD_qgr_res.pickle", "rb"))
comb_clun_isosub_filt_expvar=pd.read_pickle("./../comb_clun_isosub_filt_expvar.pkl")

In [44]:
#select isoSAAV among HPP-pass peptides
a=comb_datasets[comb_datasets.database_peptide.isin(comb_datasets_ncun_pepfilt.database_peptide.tolist())]
comb_datasets_ncun_pepfilt_psmisosub=a[a.isSubstitution]

In [59]:
#descriptive stats

print("number of proteotypic peptides with isoSAAV",len(comb_datasets_ncun_pepfilt_psmisosub.database_peptide.unique()))
print("number of proteins with isoSAAV",len(comb_datasets_ncun_pepfilt_psmisosub.leadprot.unique()))
print("number of gnomAD-SAAVs", comb_clun_isosub_filt_expvar[comb_clun_isosub_filt_expvar.database_peptide.isin(
    comb_datasets_ncun_pepfilt_psmisosub.database_peptide.unique()
)].isSAAV_tmp.value_counts())
print("number of peptides with gnomAD-SAAVs", len(comb_clun_isosub_filt_expvar[comb_clun_isosub_filt_expvar.database_peptide.isin(
    comb_datasets_ncun_pepfilt_psmisosub.database_peptide.unique())&(comb_clun_isosub_filt_expvar.isSAAV_tmp)].database_peptide.unique()))

number of proteotypic peptides with isoSAAV 68
number of proteins with isoSAAV 43
number of gnomAD-SAAVs isSAAV_tmp
False    1970
True        6
Name: count, dtype: int64
number of peptides with gnomAD-SAAVs 4


In [55]:
#map variant frequency

a=comb_clun_isosub_filt_expvar[comb_clun_isosub_filt_expvar.database_peptide.isin(
    comb_datasets_ncun_pepfilt_psmisosub.database_peptide.unique())&(comb_clun_isosub_filt_expvar.isSAAV_tmp)]

a[['chromosome','triplet','ALT', 'REF']].head()
a["af"]=a.apply(lambda x:  [ variant['exome']['af'] if isinstance(variant['exome'],dict) 
                            else  variant['genome']['af'] if isinstance(variant['genome'],dict) 
                            else 0
                for variant in gnomAD_qgr_res_uniq[str(x['chromosome'])][int(x['triplet'])] 
                if f"{x['chromosome']}-{x['triplet']}-{x['REF']}-{x['ALT']}" in variant['variant_id']][0], axis=1)
a[['chromosome','triplet','ALT', 'REF','af']].af.max()

#create file with nessesary detaild for ms2pip predictions of alternative sequences for peptides with SAAV
#match alternative sequences to psm level (get all psms that have corresponding isoSAAV)
#add modifications to alternative peptide, except the isoSAAV modification
#by psm
b=a.explode(['dataset','spectrum_title',
       'scan', 'spectrum_file', 'observed_retention_time', 'charge',
       'modifications_masses'])

#where the SAAV is on pep level
b["tosub_aa"]=b.apply(lambda x: [a for i,(a,b) in enumerate(zip(x["database_peptide"],x["peptides_w1sub"])) if a!=b][0],axis=1)

#which of psms has target mass shifts
#find mass-shifts of same aa that was substituted and verify that this mass shift can lead to sub

b["hasIsoSAAAV"]=b.apply(lambda x: any([x["tosub_aa"] in unimod_df[unimod_df.mono_mass==mass].residues.tolist() and mass in unimod_df_subs.mono_mass.tolist() for pos,aa,mass in x["modifications_masses"]])
        if x["modifications_masses"]!='Unmodified' else False, axis=1)

print("n psms holding isoSAAV~SAAV", b.hasIsoSAAAV.value_counts())

#continue further only with hasIsoSAAAV==True

b=b[b.hasIsoSAAAV]

#fix nans in mass-shifts

#bring back ionbot annotation of modifications
b=b.merge(comb_datasets[["modifications","spectrum_title"]], on="spectrum_title",how="inner")
mask=[i  for i, row in b.iterrows() if "999" in row["modifications"]]
b.loc[mask,"modifications_masses"]=b.loc[mask].apply(lambda x: [(mod[0],mod[1],
                   round([v for k,v in unimod_fix_mass.items() if k in x["modifications"]][0],4)) 
                  if pd.isna(mod[2]) else mod
       for mod in x["modifications_masses"]], axis=1)

#as we don't know the location of modification that should be masked with saav, we will try all of them
#(concidering typical aa positions of modifications) and concider the best match according to MS2PIP as closest to possible SAAV
#new plan - instead of all combinations, subtract the isoSub mass-shidt once from each ionbot predicted position + both
c=b.copy(deep=True)

def get_new_modification_mass(x):
    sub_masses=set([mass for pos,aa,mass in x["modifications_masses"] if mass in unimod_df_subs.mono_mass.tolist()])
    combs=[x["modifications_masses"]]
    for submass in sub_masses:
        n_submass=len([1 for pos,aa,mass in x["modifications_masses"] if mass==submass])
        #print(x["modifications_masses"],n_submass)
        for i in range(n_submass):
            #print(i)
            j=0; comb=[]
            for jj,(pos,aa,mass) in enumerate(x["modifications_masses"]):
                #print(j,(pos,aa,mass))
                if mass==submass and j==i: j+=1; continue #skip one pos of submass
                if mass==submass: j+=1
                #print("add")
                comb.append((pos,aa,mass))                
            combs.append(comb)
    return combs

c["PTM_combs"]=c.apply(lambda x: get_new_modification_mass(x),axis=1)
c=c.explode("PTM_combs")

#prepare alternative peptide to visualisation
c["peptide_spectrum_utils"]=c.apply(
                                    lambda x: get_modpep(x,mode="spectrum_utils"), axis=1)
c["peptides_w1sub_spectrum_utils"]=c.apply(
                                    lambda x: get_modpep(x, pep_col="peptides_w1sub", mod_mas_col="PTM_combs",mode="spectrum_utils"), axis=1)

comb_datasets_ncpsm_altpepcomb=c.copy(deep=True)
comb_datasets_ncpsm_altpepcomb["spectrum_file_mzxml"]=comb_datasets_ncpsm_altpepcomb.spectrum_file.apply(lambda x: x.replace("mgf","mzXML"))
#nosaveyet####comb_datasets_ncpsm_altpepcomb.to_pickle("comb_datasets_ncpsm_altpepcomb.pkl")

/tmp/ipykernel_3067569/3394625111.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  a["af"]=a.apply(lambda x:  [ variant['exome']['af'] if isinstance(variant['exome'],dict)


n psms holding isoSAAV~SAAV hasIsoSAAAV
True     23
False     2
Name: count, dtype: int64


In [58]:
#indicate on peptide level, as all isoSAAV PEPTIDES with gnomAD variant are further filtered out

comb_datasets_ncun_pepfilt["hasIsoSAAV"]=comb_datasets_ncun_pepfilt.database_peptide.apply(lambda x: x in comb_datasets_ncpsm_altpepcomb.database_peptide.tolist())
print("number of peptides holding isoSAAV~SAAV",comb_datasets_ncun_pepfilt["hasIsoSAAV"].value_counts())

#caution, 1 protein may have 1 isoSAAV~SAAV peptide and 1 non-isoSAAV~SAAV peptide

number of peptides holding isoSAAV~SAAV hasIsoSAAV
False    88
True      4
Name: count, dtype: int64
number of proteins WITHOUT isoSAAV~SAAV 44


#### Proteotypic peptides with isoSAAV explained by other peptide in database

In [70]:
#check that substitution location is on expected residue

comb_datasets_ncun_pepfilt_psmisosub["isSubAA"]=np.nan
comb_datasets_ncun_pepfilt_psmisosub["isSubMass"]=np.nan
for i, row in comb_datasets_ncun_pepfilt_psmisosub[comb_datasets_ncun_pepfilt_psmisosub.isSubstitution].iterrows():
    modifications_masses=row.modifications_masses
    for pos, aa, mass in modifications_masses:
        info=unimod_df_subs[unimod_df_subs.mono_mass==mass]
        if len(info[info.residues.str.contains(aa)]):
            comb_datasets_ncun_pepfilt_psmisosub.loc[i,["isSubAA","isSubMass"]]=[aa,mass]
print("number of peptides with expected PTM-residue pair",comb_datasets_ncun_pepfilt_psmisosub[comb_datasets_ncun_pepfilt_psmisosub.isSubstitution].isSubAA.isna().value_counts())

number of peptides with expected PTM-residue pair isSubAA
True     138
False     16
Name: count, dtype: int64


In [75]:
#however, we are not sure if location is correct, so get mass and compare it to shared peptides of same mass with 1-aa distance
# mass is already available in column peptide_mass (includes mass shift)

mass_error=0.000100
for i, row in comb_datasets_ncun_pepfilt_psmisosub[comb_datasets_ncun_pepfilt_psmisosub.isSubstitution].iterrows():
    peptide_length=len(row.database_peptide)
    peptide_mass=row.peptide_mass
    candidates=insituPEP.loc[((insituPEP.peptide!=row.database_peptide)&(insituPEP.peptide_length==peptide_length)&(abs(insituPEP.peptide_mass-peptide_mass)<=mass_error))]
    #dedupl
    candidates.drop_duplicates("peptide",inplace=True)
    #check for 1 aa distance
    ham=1/peptide_length
    candidates["1_aa_dist"]=candidates.peptide.apply(lambda x: distance.hamming(list(x),list(row.database_peptide))<=ham)
    comb_datasets_ncun_pepfilt_psmisosub.loc[i,["1_aa_dist"]]="|".join(candidates[candidates["1_aa_dist"]].peptide.tolist()) if len(candidates[candidates["1_aa_dist"]])>0 else False

/tmp/ipykernel_3067569/2550274903.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidates.drop_duplicates("peptide",inplace=True)
/tmp/ipykernel_3067569/2550274903.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidates["1_aa_dist"]=candidates.peptide.apply(lambda x: distance.hamming(list(x),list(row.database_peptide))<=ham)
/tmp/ipykernel_3067569/2550274903.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide

In [76]:
#how many non-canonical proteorypic peptides with isoSAAV can be explained by canonical peptide in database
comb_datasets_ncun_pepfilt_psmisosub.loc[comb_datasets_ncun_pepfilt_psmisosub.isSubstitution][["1_aa_dist"]].value_counts()

1_aa_dist      
False              141
FESPEVAER            3
KVIDDTNITR           1
LALDVEIATYRK         1
LQAEIEGLKGQR         1
LTLDKLDVK            1
LTSLAAALDENKDGK      1
NLPFDFTWK            1
RTEMENEFVLIKK        1
TLNNKFASFIDK         1
TQEKEQIKTLNNK        1
TTTGNKVFGALK         1
Name: count, dtype: int64

In [77]:
# how many peptides had non-expected residue location with substitution, but found SAAV canonical peptide
comb_datasets_ncun_pepfilt_psmisosub[(comb_datasets_ncun_pepfilt_psmisosub.isSubstitution)&(
    comb_datasets_ncun_pepfilt_psmisosub["1_aa_dist"]!=False)&(comb_datasets_ncun_pepfilt_psmisosub.isSubAA.isna())][["database_peptide","modifications_masses","1_aa_dist"]]
#acetylation, with substitutions in S->T ; G->A ; V->I ; but different location
#how do we treat this instanses??? 
#If location is predicted diferent, but peptide mass is very similar, then it is a question of location prediction quality aka spectra qc

,database_peptide,modifications_masses,1_aa_dist
205424,SLNNKFASFIDK,"[(2, I, 14.0157)]",TLNNKFASFIDK
252965,LGLDVEIATYRK,"[(0, N-term, 14.0157)]",LALDVEIATYRK
292396,KVIDDTNVTR,"[(6, T, 14.0157)]",KVIDDTNITR


### Merge peptide and psm level filtering results

In [124]:
def filter_and_check(grp):
    # subset dataframe rows for current group
    subdf = comb_datasets_ncun_psm.loc[grp.index]

    # condition inside the group
    mask =  (subdf.hasIsoSAAV == True) | (~subdf['1_aa_dist'].apply(lambda x: isinstance(x, str)))

    # filter subdf by mask
    filtered = subdf.loc[mask]

    return check_2_pep(filtered)


In [125]:
comb_datasets_ncun_psm = comb_datasets_ncun.copy()

# Precompute peptide sets for fast lookup
pep_set = set(comb_datasets_ncun_pepfilt.database_peptide)
pep_set_saav = set(comb_datasets_ncun_pepfilt[comb_datasets_ncun_pepfilt.hasIsoSAAV].database_peptide)

# Mark peptides left after HPP filtering
comb_datasets_ncun_psm["hpp_isProt2Pep"] = comb_datasets_ncun_psm["database_peptide"].isin(pep_set)

# Mark peptides with isoSAAV~gnomAD-SAAV
comb_datasets_ncun_psm["hasIsoSAAV"] = comb_datasets_ncun_psm["database_peptide"].isin(pep_set_saav)

# Merge isoSAAV explained by other peptide in database
comb_datasets_ncun_psm = comb_datasets_ncun_psm.merge(
    comb_datasets_ncun_pepfilt_psmisosub[["scan", "ionbot_match_id", "spectrum_file", "1_aa_dist"]],
    on=["scan", "ionbot_match_id", "spectrum_file"],
    how="left"
)

# Check at least 2 peptides per protein
comb_datasets_ncun_psm["hppisosaav_isProt2Pep"] = comb_datasets_ncun_psm.groupby("leadprot")["leadprot"].transform(filter_and_check) & comb_datasets_ncun_psm.hpp_isProt2Pep

#

#track - after isoSAAV+2pep check
hpp_countdown=track_hpp_countdown(hpp_countdown, "2pep2_isosaav", comb_datasets_ncun_psm[comb_datasets_ncun_psm["hppisosaav_isProt2Pep"]].drop_duplicates("database_peptide"), comb_datasets_ncun)

### Compile the table for spectra visualisation of all HPP-pass peptides

In [127]:
#select best scoring
comb_datasets_ncun_psm["isBest"] = False
best_idx = comb_datasets_ncun_psm.sort_values("psm_score", ascending=False).groupby("database_peptide").head(1).index
comb_datasets_ncun_psm.loc[best_idx, "isBest"] = True

In [132]:
#how many best PSMs with unidentified unimod masses 
comb_datasets_ncun_psm["mass_nan"]=comb_datasets_ncun_psm.modifications_masses.apply(lambda x : any(pd.isna(mod[2]) for mod in x) if x!="Unmodified" else False)
comb_datasets_ncun_psm[["isBest","mass_nan"]].value_counts()

isBest  mass_nan
False   False       2988
True    False       1016
False   True          91
True    True          37
Name: count, dtype: int64

In [196]:
#Fix unidentified unimod masses for best-scooring PSMs
comb_datasets_ncun_psm.loc[(comb_datasets_ncun_psm.isBest)&(comb_datasets_ncun_psm.mass_nan),"modifications_masses"]=comb_datasets_ncun_psm[(comb_datasets_ncun_psm.isBest)&(comb_datasets_ncun_psm.mass_nan)].apply(lambda x: [(mod[0],mod[1],
                                                                                                   round([v for k,v in unimod_fix_mass.items() if k in x["modifications"]][0],4)) 
                                                                                                  if pd.isna(mod[2]) else mod
                                                                                       for mod in x["modifications_masses"]], axis=1)

#compose USI

comb_datasets_ncun_psm["USI"]="mzspec:"+comb_datasets_ncun_psm.dataset.apply(lambda x: x.split(".")[0])+":"+comb_datasets_ncun_psm.spectrum_file.apply(lambda x: x.replace("mgf","raw"))+":scan:"+comb_datasets_ncun_psm.scan.astype(str)+":"+comb_datasets_ncun_psm.apply(
                                    lambda x: get_modpep(x, mode="usi"), axis=1)+"/"+comb_datasets_ncun_psm.charge.astype(str)

#fix USI 
comb_datasets_ncun_psm.loc[(comb_datasets_ncun_psm.isBest)&(comb_datasets_ncun_psm.mass_nan),"USI"]=comb_datasets_ncun_psm[(comb_datasets_ncun_psm.isBest)&(comb_datasets_ncun_psm.mass_nan)].apply(lambda x: x["USI"].replace(
                                                                                                [k for k,v in unimod_fix.items() if k in x["USI"]][0],
                                                                                                [v for k,v in unimod_fix.items() if k in x["USI"]][0]
                                                                                            ), axis=1)

#compose modified peptide for visualisation

comb_datasets_ncun_psm["peptide_spectrum_utils"]=comb_datasets_ncun_psm.apply(
                                    lambda x: get_modpep(x, mode="spectrum_utils"), axis=1)

#correct spectrum_file names to match mzxml

comb_datasets_ncun_psm["spectrum_file_mzxml"]=comb_datasets_ncun_psm.spectrum_file.apply(lambda x: x.replace("mgf","mzXML"))

In [197]:
comb_datasets_ncun_psm.to_csv("ionbot_open_noncanon_psm_hpp_pass_4.csv")

In [198]:
good_spectra_manual=np.nan
hpp_countdown["psm"]["manualspec"]=good_spectra_manual
hpp_countdown["pep"]["manualspec"]=good_spectra_manual
hpp_countdown["prot"]["manualspec"]=good_spectra_manual

### Visualise filtering steps

In [200]:
hpp_countdown_df = pd.DataFrame(hpp_countdown)
hpp_countdown_df.index.name = 'step'
hpp_countdown_df

,psm,pep,prot
step,,,
original,4132.0,1053.0,797.0
<9aa,3971.0,996.0,750.0
nested,2811.0,853.0,750.0
<18aa,2802.0,851.0,749.0
uniqness,1320.0,573.0,525.0
2pep1,231.0,92.0,44.0
2pep2_isosaav,219.0,82.0,39.0
manualspec,NaN,NaN,NaN


In [203]:
print("",hpp_countdown_df["pep"].original-hpp_countdown_df["pep"]["2pep1"])
print("",(hpp_countdown_df["pep"]["<18aa"]-hpp_countdown_df["pep"]["uniqness"])/(hpp_countdown_df["pep"].original-hpp_countdown_df["pep"]["2pep2_isosaav"]))
print("",(hpp_countdown_df["pep"]["2pep1"]-hpp_countdown_df["pep"]["2pep2_isosaav"])/(hpp_countdown_df["pep"].original-hpp_countdown_df["pep"]["2pep2_isosaav"]))
print(hpp_countdown_df["prot"]/hpp_countdown_df.loc["original","prot"])

 961.0
 0.286302780638517
 0.010298661174047374
step
original         1.000000
<9aa             0.941029
nested           0.941029
<18aa            0.939774
uniqness         0.658720
2pep1            0.055207
2pep2_isosaav    0.048934
manualspec            NaN
Name: prot, dtype: float64


In [ ]:
sankey_stepwise(hpp_countdown_df, 'psm',figsave_folder,"hpp4")
sankey_stepwise(hpp_countdown_df, 'pep',figsave_folder,"hpp4")
sankey_stepwise(hpp_countdown_df, 'prot',figsave_folder,"hpp4")

## Compile summary tables

## how many proteins out of detected could be theoreticaly pass HPP?